# 📈 Logistic Regression Trading Strategy: Beating Buy-the-Dip
## Stocks: NVDA & MSFT | Hold Period: 1 Week – 1 Month

This notebook builds a **multi-signal logistic regression trading strategy** using:
- 🔧 Technical Indicators (RSI, MACD, Bollinger Bands, Moving Averages)
- 📊 Fundamental Indicators (P/E ratio, EPS growth, Revenue)
- 💬 Sentiment Analysis (News sentiment scoring)
- 🚀 Momentum Indicators (Rate of Change, Price Momentum)
- 🌍 Macroeconomic & Geopolitical Variables (Interest Rates, VIX, DXY, Inflation)

**Goal:** Generate Buy/Sell signals via Logistic Regression and backtest against a naive Buy-the-Dip baseline.

## 1. Import Required Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Core
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Financial Data
import yfinance as yf

# Technical Analysis
import ta
from ta.trend import MACD, SMAIndicator, EMAIndicator
from ta.momentum import RSIIndicator, ROCIndicator
from ta.volatility import BollingerBands, AverageTrueRange

# Sentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Machine Learning
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.pipeline import Pipeline

# Visualization
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

print("✅ All libraries imported successfully!")
print(f"📅 Analysis Date: {datetime.today().strftime('%Y-%m-%d')}")

## 2. Select Stock Universe & Configuration

In [ ]:
# ─── Stock Universe ────────────────────────────────────────────────────────────
STOCKS = ["NVDA", "MSFT"]

# ─── Date Range ────────────────────────────────────────────────────────────────
END_DATE   = datetime.today().strftime('%Y-%m-%d')
START_DATE = (datetime.today() - timedelta(days=5*365)).strftime('%Y-%m-%d')

# ─── Strategy Config ───────────────────────────────────────────────────────────
HOLD_PERIODS = {
    "1W":  5,    # 1 week  in trading days
    "2W":  10,   # 2 weeks in trading days
    "1M":  21,   # 1 month in trading days
}
TARGET_HOLD = "1W"        # Primary hold period to optimise for
THRESHOLD   = 0.0025      # Min return threshold to classify as "Buy" (25bps)

# ─── Macro Tickers ─────────────────────────────────────────────────────────────
MACRO_TICKERS = {
    "VIX":  "^VIX",        # Fear / volatility index
    "DXY":  "DX-Y.NYB",    # US Dollar index
    "TNX":  "^TNX",        # 10-Year Treasury yield
    "OIL":  "CL=F",        # Crude Oil futures
    "SPY":  "SPY",         # S&P 500 ETF (broad market)
}

print(f"📋 Stocks selected  : {STOCKS}")
print(f"📅 Date range       : {START_DATE}  →  {END_DATE}")
print(f"⏱  Hold periods     : {HOLD_PERIODS}")
print(f"🎯 Primary hold     : {TARGET_HOLD} ({HOLD_PERIODS[TARGET_HOLD]} trading days)")
print(f"📈 Buy threshold    : >{THRESHOLD*100:.2f}% forward return")

## 3. Fetch Historical Stock & Macro Data

In [ ]:
def fetch_ohlcv(ticker: str, start: str, end: str) -> pd.DataFrame:
    """Download OHLCV data and clean it up (handles yfinance MultiIndex columns)."""
    df = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    # yfinance ≥0.2 returns MultiIndex columns like ('Close','NVDA')
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0].lower() for col in df.columns]
    else:
        df.columns = [c.lower() for c in df.columns]
    df.index = pd.to_datetime(df.index)
    df.dropna(inplace=True)
    return df

# ── Stock price data ───────────────────────────────────────────────────────────
raw_stock = {}
for ticker in STOCKS:
    raw_stock[ticker] = fetch_ohlcv(ticker, START_DATE, END_DATE)
    print(f"  {ticker}: {len(raw_stock[ticker])} rows | "
          f"{raw_stock[ticker].index[0].date()} → {raw_stock[ticker].index[-1].date()}")

# ── Macro / risk-factor data ───────────────────────────────────────────────────
raw_macro = {}
print("\nFetching macro data …")
for name, ticker in MACRO_TICKERS.items():
    df = fetch_ohlcv(ticker, START_DATE, END_DATE)
    raw_macro[name] = df[['close']].rename(columns={'close': name})
    print(f"  {name} ({ticker}): {len(raw_macro[name])} rows")

# Merge macro series into a single DataFrame aligned to trading days
macro_df = pd.concat(raw_macro.values(), axis=1).ffill()
print(f"\n✅ Macro DataFrame shape: {macro_df.shape}")

# Quick sanity plot
fig, axes = plt.subplots(1, len(STOCKS), figsize=(16, 4))
for ax, ticker in zip(axes, STOCKS):
    raw_stock[ticker]['close'].plot(ax=ax, title=f"{ticker} – Close Price", color='steelblue', linewidth=1.2)
    ax.set_xlabel("Date"); ax.set_ylabel("Price (USD)")
plt.suptitle("Historical Close Prices", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Calculate Technical Indicators

We compute a rich set of technical signals:
| Category | Indicators |
|---|---|
| Trend | SMA(10,20,50), EMA(12,26), Price vs SMA ratio |
| Momentum | RSI(14), ROC(5,10), MACD line & signal |
| Volatility | Bollinger Band width, %B, ATR(14) |
| Volume | OBV, Volume ratio (vs 20-day avg) |

In [ ]:
def add_technical_indicators(df: pd.DataFrame) -> pd.DataFrame:
    """Compute a comprehensive set of technical features."""
    close  = df['close']
    high   = df['high']
    low    = df['low']
    volume = df['volume']

    # ── Trend ──────────────────────────────────────────────────────────────────
    df['sma_10']  = SMAIndicator(close, window=10).sma_indicator()
    df['sma_20']  = SMAIndicator(close, window=20).sma_indicator()
    df['sma_50']  = SMAIndicator(close, window=50).sma_indicator()
    df['ema_12']  = EMAIndicator(close, window=12).ema_indicator()
    df['ema_26']  = EMAIndicator(close, window=26).ema_indicator()

    # Price position relative to moving averages (normalised)
    df['price_vs_sma20']  = (close - df['sma_20'])  / df['sma_20']
    df['price_vs_sma50']  = (close - df['sma_50'])  / df['sma_50']
    df['ema_crossover']   = df['ema_12'] - df['ema_26']   # positive = bullish

    # ── Momentum ───────────────────────────────────────────────────────────────
    df['rsi_14']   = RSIIndicator(close, window=14).rsi()
    df['roc_5']    = ROCIndicator(close, window=5).roc()
    df['roc_10']   = ROCIndicator(close, window=10).roc()

    macd_obj       = MACD(close, window_slow=26, window_fast=12, window_sign=9)
    df['macd']     = macd_obj.macd()
    df['macd_sig'] = macd_obj.macd_signal()
    df['macd_hist']= macd_obj.macd_diff()    # histogram = macd - signal

    # ── Volatility ─────────────────────────────────────────────────────────────
    bb             = BollingerBands(close, window=20, window_dev=2)
    df['bb_width'] = (bb.bollinger_hband() - bb.bollinger_lband()) / bb.bollinger_mavg()
    df['bb_pct']   = bb.bollinger_pband()    # %B: 0=lower, 1=upper band

    atr_obj        = AverageTrueRange(high, low, close, window=14)
    df['atr_14']   = atr_obj.average_true_range()
    df['atr_pct']  = df['atr_14'] / close    # normalise ATR by price

    # ── Volume ─────────────────────────────────────────────────────────────────
    df['vol_ratio']= volume / volume.rolling(20).mean()   # relative volume
    # On-Balance Volume (manual – faster than ta for large sets)
    obv = (np.sign(close.diff()) * volume).fillna(0).cumsum()
    df['obv_change'] = obv.pct_change(5)                  # 5-day OBV change

    # ── Daily returns & realised volatility ───────────────────────────────────
    df['ret_1d']    = close.pct_change(1)
    df['ret_5d']    = close.pct_change(5)
    df['rvol_10d']  = df['ret_1d'].rolling(10).std() * np.sqrt(252)

    return df

# Apply to all stocks
tech_data = {}
for ticker in STOCKS:
    df_tech = raw_stock[ticker].copy()
    df_tech = add_technical_indicators(df_tech)
    tech_data[ticker] = df_tech
    print(f"  {ticker}: {df_tech.shape[1]} features computed")

# ── Correlation heatmap of technical features (NVDA example) ──────────────────
tech_cols = [c for c in tech_data['NVDA'].columns
             if c not in ['open','high','low','close','volume']]
corr = tech_data['NVDA'][tech_cols].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdYlGn', center=0,
            annot=False, linewidths=0.3, ax=ax)
ax.set_title("Technical Feature Correlation – NVDA", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("✅ Technical indicators computed for all stocks.")

## 5. Add Fundamental Indicators

We pull key fundamental metrics from `yfinance` (TTM/quarterly): P/E, Forward P/E, P/S, P/B, EPS TTM, Revenue growth, Profit margin, Debt/Equity.  
These are static snapshots – they will be forward-filled across trading days (realistic for a retail-level model).

In [ ]:
def fetch_fundamentals(ticker: str) -> dict:
    """Fetch current fundamental snapshot from yfinance."""
    info = yf.Ticker(ticker).info
    fundamentals = {
        'pe_ratio':          info.get('trailingPE',          np.nan),
        'forward_pe':        info.get('forwardPE',           np.nan),
        'ps_ratio':          info.get('priceToSalesTrailing12Months', np.nan),
        'pb_ratio':          info.get('priceToBook',         np.nan),
        'eps_ttm':           info.get('trailingEps',         np.nan),
        'eps_forward':       info.get('forwardEps',          np.nan),
        'revenue_growth':    info.get('revenueGrowth',       np.nan),   # YoY
        'earnings_growth':   info.get('earningsGrowth',      np.nan),   # YoY
        'profit_margin':     info.get('profitMargins',       np.nan),
        'debt_to_equity':    info.get('debtToEquity',        np.nan),
        'return_on_equity':  info.get('returnOnEquity',      np.nan),
        'current_ratio':     info.get('currentRatio',        np.nan),
        'beta':              info.get('beta',                 np.nan),
    }
    return fundamentals

fund_snapshots = {}
for ticker in STOCKS:
    fund_snapshots[ticker] = fetch_fundamentals(ticker)
    print(f"\n📊 {ticker} Fundamentals:")
    for k, v in fund_snapshots[ticker].items():
        print(f"   {k:22s}: {v}")

# Broadcast fundamentals as constant columns across the full date index
# (In production you would use point-in-time financials from e.g. Simfin/Polygon)
fund_data = {}
for ticker in STOCKS:
    df = tech_data[ticker].copy()
    for col, val in fund_snapshots[ticker].items():
        df[col] = val   # forward-filled constant (snapshot)
    fund_data[ticker] = df

print("\n✅ Fundamental features added to all stock DataFrames.")

## 6. Incorporate Stock Sentiment Data

We use two proxies for market sentiment:
1. **VADER Sentiment** on recent news headlines fetched from Yahoo Finance RSS  
2. **Implied Sentiment from Options** – we approximate via Put/Call ratio proxy using VIX level and changes  

> 📌 **Note:** For a production system, integrate premium news APIs (e.g., Refinitiv, NewsAPI, Bloomberg). Here we simulate a sentiment time series using a realistic noise model anchored to actual price momentum, plus a live fetch of Yahoo Finance news when available.

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def score_headline(text: str) -> float:
    """Return VADER compound sentiment score in [-1, 1]."""
    return analyzer.polarity_scores(str(text))['compound']

def fetch_yahoo_news_sentiment(ticker: str, n_articles: int = 30) -> float:
    """Fetch recent Yahoo Finance headlines and return avg sentiment."""
    try:
        news = yf.Ticker(ticker).news or []
        scores = []
        for article in news[:n_articles]:
            title = article.get('content', {}).get('title', '') or article.get('title', '')
            if title:
                scores.append(score_headline(title))
        return float(np.mean(scores)) if scores else 0.0
    except Exception:
        return 0.0

# ── Live sentiment snapshot (last ~30 headlines) ──────────────────────────────
live_sentiment = {}
for ticker in STOCKS:
    s = fetch_yahoo_news_sentiment(ticker)
    live_sentiment[ticker] = s
    label = "🟢 Positive" if s > 0.05 else ("🔴 Negative" if s < -0.05 else "⚪ Neutral")
    print(f"  {ticker} live news sentiment: {s:+.4f}  {label}")

# ── Simulated historical daily sentiment series ────────────────────────────────
# Anchored to: 0.4 × sign(5d return) + 0.6 × N(0, 0.15) — realistic proxy
np.random.seed(42)
sentiment_data = {}
for ticker in STOCKS:
    df = fund_data[ticker].copy()
    ret5 = df['ret_5d'].fillna(0)
    # Lagged one day to avoid look-ahead
    sentiment_series = (0.4 * np.sign(ret5.shift(1)).fillna(0)
                        + 0.6 * np.random.normal(0, 0.15, len(df)))
    sentiment_series = sentiment_series.clip(-1, 1)

    # Smooth with a 3-day EMA to mimic news persistence
    df['sentiment_raw']   = sentiment_series
    df['sentiment_ema3']  = sentiment_series.ewm(span=3).mean()

    # Rolling 5-day sentiment momentum (is it improving or deteriorating?)
    df['sentiment_mom5']  = df['sentiment_ema3'].diff(5)

    # Add the live snapshot as the most recent override
    df.loc[df.index[-1], 'sentiment_raw']  = live_sentiment[ticker]
    df.loc[df.index[-1], 'sentiment_ema3'] = live_sentiment[ticker]

    sentiment_data[ticker] = df

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(len(STOCKS), 1, figsize=(16, 6), sharex=True)
for ax, ticker in zip(axes, STOCKS):
    s = sentiment_data[ticker]
    ax.fill_between(s.index, s['sentiment_ema3'],
                    where=s['sentiment_ema3'] >= 0, color='seagreen', alpha=0.5, label='Positive')
    ax.fill_between(s.index, s['sentiment_ema3'],
                    where=s['sentiment_ema3'] < 0,  color='tomato',   alpha=0.5, label='Negative')
    ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
    ax.set_title(f"{ticker} – Sentiment Score (EMA-3)", fontweight='bold')
    ax.set_ylabel("Sentiment")
    ax.legend(loc='upper left', fontsize=8)
plt.xlabel("Date")
plt.suptitle("Historical Sentiment Proxy", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("✅ Sentiment features computed.")

## 7. Add Momentum Indicators

Beyond RSI and ROC computed earlier, we add cross-sectional and time-series momentum signals:
- **52-week high/low proximity** – how close price is to recent extremes
- **Frog-in-the-pan (FIP)** – consistency of small vs large returns
- **Trend strength** – ADX-like measure
- **Relative Strength vs SPY** – is the stock outperforming the market?

In [ ]:
spy_close = raw_macro['SPY']['SPY']

def add_momentum_features(df: pd.DataFrame, spy: pd.Series) -> pd.DataFrame:
    close = df['close']

    # ── Time-series momentum ───────────────────────────────────────────────────
    df['mom_1m']   = close.pct_change(21)     # 1-month momentum
    df['mom_3m']   = close.pct_change(63)     # 3-month momentum
    df['mom_6m']   = close.pct_change(126)    # 6-month momentum
    df['mom_12m']  = close.pct_change(252)    # 12-month momentum

    # ── 52-week high/low proximity ─────────────────────────────────────────────
    high_52w = close.rolling(252).max()
    low_52w  = close.rolling(252).min()
    df['pct_from_52w_high'] = (close - high_52w) / high_52w   # ≤ 0
    df['pct_from_52w_low']  = (close - low_52w)  / low_52w    # ≥ 0
    df['range_position_52w'] = (close - low_52w) / (high_52w - low_52w + 1e-9)

    # ── Frog-in-the-pan (FIP) – counts sign changes in daily returns ───────────
    # A trend with many small returns (few sign changes) = stronger signal
    def fip_score(rets, window=21):
        sign_changes = (np.sign(rets) != np.sign(rets.shift(1))).rolling(window).mean()
        return 1 - sign_changes   # high FIP = consistent trend direction
    df['fip_21d'] = fip_score(df['ret_1d'], 21)

    # ── Relative strength vs SPY ───────────────────────────────────────────────
    spy_ret = spy.pct_change(21).reindex(df.index).ffill()
    df['rs_vs_spy_1m'] = df['mom_1m'] - spy_ret

    # ── Acceleration: is momentum speeding up or slowing down? ────────────────
    df['mom_accel'] = df['mom_1m'] - df['mom_1m'].shift(5)

    return df

momentum_data = {}
for ticker in STOCKS:
    df = sentiment_data[ticker].copy()
    df = add_momentum_features(df, spy_close)
    momentum_data[ticker] = df

# ── Plot momentum dashboard ────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(16, 10))
momentum_cols = ['mom_1m', 'mom_3m', 'mom_6m', 'fip_21d', 'rs_vs_spy_1m', 'range_position_52w']
titles = ['1M Momentum', '3M Momentum', '6M Momentum',
          'FIP Score (21d)', 'Rel. Strength vs SPY (1M)', '52W Range Position']
colors = ['royalblue', 'darkorange', 'seagreen', 'purple', 'crimson', 'teal']

for ax, col, title, color in zip(axes.flatten(), momentum_cols, titles, colors):
    for ticker in STOCKS:
        momentum_data[ticker][col].dropna().plot(ax=ax, label=ticker, alpha=0.8, linewidth=1.1)
    ax.axhline(0, color='grey', linewidth=0.7, linestyle='--')
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle("Momentum Feature Dashboard", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("✅ Momentum features computed.")

## 8. Integrate Macroeconomic & Geopolitical Variables

| Variable | Proxy | Interpretation |
|---|---|---|
| **VIX** | CBOE VIX | Systemic fear / risk-off |
| **DXY** | US Dollar Index | Strong USD = headwind for tech earnings |
| **10Y Yield (TNX)** | ^TNX | Rate pressure on growth stocks |
| **Oil (CL=F)** | WTI crude | Inflation / geopolitical risk |
| **Yield Curve Slope** | TNX – 2Y | Recession signal |
| **SPY Trend** | SPY 50-day trend | Broad market regime |

We also derive **regime labels** (risk-on / risk-off) based on VIX level and trend.

In [ ]:
# ── Fetch 13-week T-bill yield (proxy for short end of curve) ─────────────────
try:
    tnx_2y_raw = fetch_ohlcv("^IRX", START_DATE, END_DATE)
    tnx_2y     = tnx_2y_raw[['close']].rename(columns={'close': 'IRX'})
except Exception:
    tnx_2y = pd.DataFrame({'IRX': np.nan}, index=macro_df.index)

# ── Build enhanced macro DataFrame ────────────────────────────────────────────
macro_enhanced = macro_df.copy()
macro_enhanced = macro_enhanced.join(tnx_2y, how='left').ffill()

# Derived macro features
macro_enhanced['vix_chg_5d']    = macro_enhanced['VIX'].pct_change(5)
macro_enhanced['vix_zscore']    = ((macro_enhanced['VIX'] - macro_enhanced['VIX'].rolling(63).mean())
                                    / macro_enhanced['VIX'].rolling(63).std())
macro_enhanced['dxy_chg_5d']    = macro_enhanced['DXY'].pct_change(5)
macro_enhanced['tnx_chg_5d']    = macro_enhanced['TNX'].pct_change(5)
macro_enhanced['oil_chg_5d']    = macro_enhanced['OIL'].pct_change(5)
macro_enhanced['yield_slope']   = macro_enhanced['TNX'] - macro_enhanced.get('IRX', pd.Series(0, index=macro_enhanced.index))
macro_enhanced['spy_trend']     = (macro_enhanced['SPY'] / macro_enhanced['SPY'].rolling(50).mean()) - 1

# Market regime: 1 = risk-on (VIX < 20 & rising SPY), 0 = risk-off
macro_enhanced['regime_risk_on'] = (
    (macro_enhanced['VIX'] < 20) &
    (macro_enhanced['spy_trend'] > 0)
).astype(int)

print(f"Macro feature matrix shape: {macro_enhanced.shape}")
print(f"Columns: {list(macro_enhanced.columns)}")

# ── Add macro features to each stock DataFrame ────────────────────────────────
macro_cols = list(macro_enhanced.columns)
full_data = {}
for ticker in STOCKS:
    df = momentum_data[ticker].copy()
    df = df.join(macro_enhanced[macro_cols], how='left', rsuffix='_macro').ffill()
    full_data[ticker] = df
    print(f"  {ticker} full DataFrame: {df.shape}")

# ── Macro dashboard plot ───────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
plot_macro = ['VIX', 'DXY', 'TNX', 'OIL', 'yield_slope', 'spy_trend']
titles_m   = ['VIX (Fear Index)', 'DXY (US Dollar)', '10Y Treasury Yield',
               'Oil (WTI)', 'Yield Curve (10Y–2Y)', 'SPY Trend (vs 50MA)']

for ax, col, title in zip(axes.flatten(), plot_macro, titles_m):
    macro_enhanced[col].dropna().plot(ax=ax, color='navy', linewidth=1.0, alpha=0.85)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('')

plt.suptitle("Macroeconomic & Geopolitical Variables", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("✅ Macro features integrated.")

## 9. Prepare Features & Labels for Logistic Regression

**Label definition:**  
`y = 1 (Buy)` if the stock's forward return over the holding period exceeds the threshold `THRESHOLD` (25 bps).  
`y = 0 (Sell/Flat)` otherwise.

We build a master feature set per stock, drop NaNs, and apply a `TimeSeriesSplit` cross-validator to avoid look-ahead bias.

In [ ]:
# ── Feature columns to use ────────────────────────────────────────────────────
FEATURE_COLS = [
    # Technical
    'rsi_14', 'macd', 'macd_sig', 'macd_hist',
    'bb_width', 'bb_pct', 'atr_pct',
    'price_vs_sma20', 'price_vs_sma50', 'ema_crossover',
    'roc_5', 'roc_10', 'vol_ratio', 'obv_change', 'rvol_10d',
    # Momentum
    'mom_1m', 'mom_3m', 'mom_6m', 'mom_12m',
    'fip_21d', 'rs_vs_spy_1m', 'mom_accel',
    'pct_from_52w_high', 'pct_from_52w_low', 'range_position_52w',
    # Fundamental
    'pe_ratio', 'forward_pe', 'ps_ratio', 'pb_ratio',
    'revenue_growth', 'earnings_growth', 'profit_margin',
    'debt_to_equity', 'return_on_equity', 'beta',
    # Sentiment
    'sentiment_raw', 'sentiment_ema3', 'sentiment_mom5',
    # Macro / Geopolitical
    'VIX', 'vix_chg_5d', 'vix_zscore',
    'DXY', 'dxy_chg_5d',
    'TNX', 'tnx_chg_5d',
    'oil_chg_5d', 'yield_slope', 'spy_trend',
    'regime_risk_on',
]

def build_ml_dataset(df: pd.DataFrame, hold_days: int, threshold: float):
    """Construct (X, y, dates) for a given holding period."""
    df = df.copy()

    # Forward return over the holding window (shift back to avoid look-ahead)
    df['fwd_return'] = df['close'].pct_change(hold_days).shift(-hold_days)
    df['label']      = (df['fwd_return'] > threshold).astype(int)

    # Keep only rows where all feature columns exist
    available_cols = [c for c in FEATURE_COLS if c in df.columns]
    data = df[available_cols + ['label', 'fwd_return']].dropna()

    X      = data[available_cols]
    y      = data['label']
    dates  = data.index
    fwd_r  = data['fwd_return']
    return X, y, dates, fwd_r, available_cols

# Build datasets for each stock & each hold period
ml_datasets = {}
for ticker in STOCKS:
    ml_datasets[ticker] = {}
    for period_name, hold_days in HOLD_PERIODS.items():
        X, y, dates, fwd_r, feat_cols = build_ml_dataset(
            full_data[ticker], hold_days, THRESHOLD)
        ml_datasets[ticker][period_name] = {
            'X': X, 'y': y, 'dates': dates,
            'fwd_return': fwd_r, 'feature_cols': feat_cols
        }
        buy_pct = y.mean() * 100
        print(f"  {ticker} | {period_name}: {len(X)} samples | "
              f"Buy label = {buy_pct:.1f}% | Features = {len(feat_cols)}")

print(f"\n✅ ML datasets constructed for all stocks and hold periods.")

## 10. Train Logistic Regression Model

We use **TimeSeriesSplit** (5 folds, no shuffling) to preserve temporal order and avoid leakage.  
A `StandardScaler → LogisticRegression(L2)` pipeline is fitted on training folds and evaluated on the held-out future fold.  
We report: **Accuracy, AUC-ROC, Precision, Recall, F1** and plot the confusion matrix.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

N_SPLITS   = 5
tscv       = TimeSeriesSplit(n_splits=N_SPLITS)

def train_evaluate_lr(ticker: str, period: str) -> dict:
    """Train & cross-validate LR model, return metrics and final fitted model."""
    ds   = ml_datasets[ticker][period]
    X, y = ds['X'].values, ds['y'].values

    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('lr',     LogisticRegression(
                       C=0.5,                # L2 regularisation
                       class_weight='balanced',
                       max_iter=1000,
                       random_state=42))
    ])

    fold_results = []
    all_y_true, all_y_pred, all_y_prob = [], [], []

    for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]

        pipe.fit(X_tr, y_tr)
        y_pred  = pipe.predict(X_te)
        y_prob  = pipe.predict_proba(X_te)[:, 1]

        try:
            auc = roc_auc_score(y_te, y_prob)
        except Exception:
            auc = np.nan

        fold_results.append({'fold': fold+1, 'auc': auc,
                             'acc': (y_pred == y_te).mean()})
        all_y_true.extend(y_te); all_y_pred.extend(y_pred); all_y_prob.extend(y_prob)

    # Final model trained on ALL data (for signal generation)
    pipe.fit(X, y)

    results = {
        'model':     pipe,
        'y_true':    np.array(all_y_true),
        'y_pred':    np.array(all_y_pred),
        'y_prob':    np.array(all_y_prob),
        'fold_df':   pd.DataFrame(fold_results),
        'feature_cols': ds['feature_cols']
    }
    return results

# ── Train for all tickers × hold periods ──────────────────────────────────────
trained_models = {}
for ticker in STOCKS:
    trained_models[ticker] = {}
    for period in HOLD_PERIODS:
        res = train_evaluate_lr(ticker, period)
        trained_models[ticker][period] = res
        avg_auc = res['fold_df']['auc'].mean()
        avg_acc = res['fold_df']['acc'].mean()
        print(f"  {ticker} | {period}  →  AUC: {avg_auc:.4f}  |  Acc: {avg_acc:.4f}")

# ── Detailed report & confusion matrix for primary hold period ────────────────
fig, axes = plt.subplots(1, len(STOCKS), figsize=(12, 4))
for ax, ticker in zip(axes, STOCKS):
    res   = trained_models[ticker][TARGET_HOLD]
    cm    = confusion_matrix(res['y_true'], res['y_pred'])
    disp  = ConfusionMatrixDisplay(cm, display_labels=['Sell/Flat', 'Buy'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    auc   = roc_auc_score(res['y_true'], res['y_prob'])
    ax.set_title(f"{ticker} | {TARGET_HOLD}  AUC={auc:.4f}", fontweight='bold')

plt.suptitle("Confusion Matrix (CV Out-of-Fold Predictions)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

for ticker in STOCKS:
    res = trained_models[ticker][TARGET_HOLD]
    print(f"\n{'='*50}")
    print(f"  {ticker} | {TARGET_HOLD} – Classification Report")
    print(f"{'='*50}")
    print(classification_report(res['y_true'], res['y_pred'],
                                target_names=['Sell/Flat', 'Buy']))

print("✅ Models trained and evaluated.")

## 11. Generate Buy/Sell Signals & Feature Importance

We plot **logistic regression coefficients** (proxy for feature importance after standardisation) and generate the full signal time-series for the test window.

In [ ]:
def plot_feature_importance(ticker: str, period: str, top_n: int = 20):
    """Plot top-N feature importances (LR coefficients after scaling)."""
    res       = trained_models[ticker][period]
    model     = res['model']
    feat_cols = res['feature_cols']
    coefs     = model.named_steps['lr'].coef_[0]

    importance = pd.Series(np.abs(coefs), index=feat_cols).sort_values(ascending=False)
    top         = importance.head(top_n)
    colors_imp  = ['tomato' if coefs[feat_cols.index(f)] < 0 else 'seagreen' for f in top.index]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(top.index[::-1], top.values[::-1], color=colors_imp[::-1])
    ax.set_xlabel("|Coefficient| (scaled)")
    ax.set_title(f"{ticker} | {period} – Top {top_n} Feature Importances\n"
                 f"(🟢 Positive / 🔴 Negative coefficient)", fontweight='bold')
    plt.tight_layout()
    plt.show()

for ticker in STOCKS:
    plot_feature_importance(ticker, TARGET_HOLD)

# ── Generate full signal series ────────────────────────────────────────────────
signal_series = {}
for ticker in STOCKS:
    ds     = ml_datasets[ticker][TARGET_HOLD]
    model  = trained_models[ticker][TARGET_HOLD]['model']
    X      = ds['X'].values
    dates  = ds['dates']

    prob_buy = model.predict_proba(X)[:, 1]
    signal   = (prob_buy > 0.5).astype(int)          # 1=Buy, 0=Sell/Flat

    sig_df = pd.DataFrame({
        'signal':   signal,
        'prob_buy': prob_buy,
        'close':    full_data[ticker].loc[dates, 'close']
    }, index=dates)

    signal_series[ticker] = sig_df

    # ── Signal overlay on price chart ─────────────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True,
                                    gridspec_kw={'height_ratios': [3, 1]})

    ax1.plot(sig_df.index, sig_df['close'], color='steelblue', linewidth=1.2, label='Price')
    buys  = sig_df[sig_df['signal'] == 1]
    sells = sig_df[sig_df['signal'] == 0]
    ax1.scatter(buys.index,  buys['close'],  marker='^', color='lime',   s=30,
                zorder=5, alpha=0.7, label='Buy signal')
    ax1.scatter(sells.index, sells['close'], marker='v', color='tomato', s=30,
                zorder=5, alpha=0.7, label='Sell signal')
    ax1.set_ylabel("Price (USD)"); ax1.legend(fontsize=9)
    ax1.set_title(f"{ticker} – Buy/Sell Signals ({TARGET_HOLD} hold) from Logistic Regression",
                  fontweight='bold')

    ax2.fill_between(sig_df.index, sig_df['prob_buy'], 0.5,
                     where=sig_df['prob_buy'] >= 0.5, color='lime',   alpha=0.6)
    ax2.fill_between(sig_df.index, sig_df['prob_buy'], 0.5,
                     where=sig_df['prob_buy'] < 0.5,  color='tomato', alpha=0.6)
    ax2.axhline(0.5, color='grey', linestyle='--', linewidth=0.8)
    ax2.set_ylabel("P(Buy)"); ax2.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()

print("✅ Buy/Sell signals generated.")

## 12. Backtest Strategy Performance

We compare three strategies:

| Strategy | Logic |
|---|---|
| **LR Signal** | Long when model predicts Buy, Flat otherwise |
| **Buy & Hold** | Long 100% of the time |
| **Buy the Dip** | Buy when 5-day return < −3%, sell after hold period |

Metrics reported: **Total Return, CAGR, Sharpe Ratio, Max Drawdown, Calmar Ratio, Hit Rate**

In [ ]:
def compute_portfolio_metrics(equity_curve: pd.Series, risk_free_rate: float = 0.05) -> dict:
    """Compute key performance metrics from an equity curve."""
    returns = equity_curve.pct_change().dropna()
    n_years = len(returns) / 252

    total_return   = equity_curve.iloc[-1] / equity_curve.iloc[0] - 1
    cagr           = (1 + total_return) ** (1 / max(n_years, 0.01)) - 1

    excess_ret     = returns - risk_free_rate / 252
    sharpe         = excess_ret.mean() / (excess_ret.std() + 1e-9) * np.sqrt(252)

    rolling_max    = equity_curve.cummax()
    drawdown       = (equity_curve - rolling_max) / rolling_max
    max_dd         = drawdown.min()

    calmar         = cagr / abs(max_dd + 1e-9)

    # Hit rate: % of trading days with positive return
    hit_rate       = (returns > 0).mean()

    return {
        'Total Return':  f"{total_return*100:.2f}%",
        'CAGR':          f"{cagr*100:.2f}%",
        'Sharpe Ratio':  f"{sharpe:.3f}",
        'Max Drawdown':  f"{max_dd*100:.2f}%",
        'Calmar Ratio':  f"{calmar:.3f}",
        'Hit Rate':      f"{hit_rate*100:.2f}%",
    }

DIP_THRESHOLD = -0.03   # 3% drop in 5 days triggers dip buy

all_metrics = {}

for ticker in STOCKS:
    sig_df  = signal_series[ticker]
    df_full = full_data[ticker].loc[sig_df.index].copy()
    daily_ret = df_full['close'].pct_change().fillna(0)

    # ── 1. LR Strategy ────────────────────────────────────────────────────────
    # Signal is shifted by 1 day (trade next open, not same close)
    lr_pos      = sig_df['signal'].shift(1).fillna(0)
    lr_returns  = (lr_pos * daily_ret)
    lr_equity   = (1 + lr_returns).cumprod()

    # ── 2. Buy & Hold ─────────────────────────────────────────────────────────
    bh_equity   = (1 + daily_ret).cumprod()

    # ── 3. Buy the Dip ────────────────────────────────────────────────────────
    ret_5d      = df_full['close'].pct_change(5)
    in_dip      = (ret_5d.shift(1) < DIP_THRESHOLD).astype(float)

    # Hold for TARGET_HOLD period after dip entry
    hold_days   = HOLD_PERIODS[TARGET_HOLD]
    dip_pos     = in_dip.copy()
    for i in range(1, hold_days):
        dip_pos = np.maximum(dip_pos, in_dip.shift(i).fillna(0))

    dip_returns = dip_pos * daily_ret
    dip_equity  = (1 + dip_returns).cumprod()

    # ── Compile equity curves ─────────────────────────────────────────────────
    equity = pd.DataFrame({
        'LR Strategy':   lr_equity,
        'Buy & Hold':    bh_equity,
        'Buy the Dip':   dip_equity,
    }).dropna()

    # ── Metrics table ─────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  {ticker} – Performance Summary ({TARGET_HOLD} hold)")
    print(f"{'='*60}")
    metrics_rows = {}
    for strat in equity.columns:
        m = compute_portfolio_metrics(equity[strat])
        metrics_rows[strat] = m
        print(f"\n  📌 {strat}")
        for k, v in m.items():
            print(f"     {k:15s}: {v}")
    all_metrics[ticker] = pd.DataFrame(metrics_rows).T

    # ── Equity curve plot ─────────────────────────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), sharex=True,
                                    gridspec_kw={'height_ratios': [3, 1]})

    colors_strat = {'LR Strategy': 'royalblue', 'Buy & Hold': 'darkorange', 'Buy the Dip': 'seagreen'}
    for strat, col in colors_strat.items():
        equity[strat].plot(ax=ax1, color=col, linewidth=1.5, label=strat)

    ax1.set_ylabel("Portfolio Value (normalised)")
    ax1.legend(fontsize=10)
    ax1.set_title(f"{ticker} – Strategy Comparison ({TARGET_HOLD} hold period)", fontweight='bold', fontsize=13)
    ax1.axhline(1.0, color='grey', linewidth=0.7, linestyle='--')

    # Drawdown
    for strat, col in colors_strat.items():
        dd = (equity[strat] - equity[strat].cummax()) / equity[strat].cummax()
        ax2.fill_between(equity.index, dd, 0, alpha=0.35, color=col, label=strat)
    ax2.set_ylabel("Drawdown")
    ax2.set_xlabel("Date")
    ax2.legend(fontsize=8, loc='lower left')

    plt.tight_layout()
    plt.show()

print("\n✅ Backtest complete.")

## 📋 Summary & Next Steps

### What this notebook does
| Step | Detail |
|---|---|
| Data | 5-year daily OHLCV for NVDA & MSFT + 5 macro tickers |
| Features | 40+ features across Technical, Fundamental, Sentiment, Momentum & Macro |
| Model | Logistic Regression (L2, balanced classes) with `TimeSeriesSplit` CV |
| Signals | Daily Buy / Sell/Flat, traded at next-day open |
| Benchmarks | Buy & Hold, Buy-the-Dip (−3% in 5 days, hold 1W) |

### Recommended Improvements
1. **Point-in-time fundamentals** – Use Simfin, Compustat or Polygon for quarterly financials
2. **Real news sentiment** – Integrate NewsAPI / Refinitiv to replace the simulated proxy
3. **Feature selection** – LASSO or RFE to prune collinear features
4. **Alternative models** – XGBoost / LightGBM for non-linear interactions
5. **Transaction costs** – Add bid-ask spread and commission to the backtest
6. **Position sizing** – Kelly criterion or vol-targeting based on `prob_buy`
7. **Expand universe** – Add more stocks; build a cross-sectional ranking model

> ⚠️ **Disclaimer:** This is for educational purposes only. Past simulated performance does not guarantee future results. Always perform rigorous out-of-sample testing before deploying real capital.